# Online learning

Start or resume an online-learning session from a laptop. The training
server receives episodes saved through the robot backend and continuously
updates the inference service.

Before starting:

- Connect to the robot network.
- Ask the backend operator to ensure the training server is running.
- Ask the backend operator to ensure the warm-start model is available on
  the training server.
- Ensure the active robot system configuration enables episode forwarding
  by setting `online_learning_server_address` to the training server.

This notebook does not start, restart, or inspect backend processes.

In [ ]:
from r2_labs.sdk import client as sdk_client
from r2_labs.sdk import rpc_api

## Configuration

Using the same model with `RESTART_TRAINING = False` resumes its existing
online-learning state. Set it to `True` to start again from the warm-start
weights with an empty online dataset; the server preserves the previous state
as a backup.

Disable the additional augmentations unless the warm-start model was trained
with the current augmentation defaults. `USE_JOINT_TORQUES` must match the
warm-start model's input structure.

In [ ]:
ROBOT_HOST: str = "robot.local"
SERVER_HOST: str = "training-server.local"
MODEL_ID: str = "skill_18_08_07_00#calm-credits-28"

INFERENCE_GPU: int = 0
INFERENCE_PORT: int = 4243
RESTART_TRAINING: bool = False
DISABLE_NEW_AUGMENTATIONS: bool = True
USE_JOINT_TORQUES: bool = True

TRAINER_ADDRESS: str = (
    f"tcp://{SERVER_HOST}:{rpc_api.DEFAULT_MODEL_TRAINER_PORT}"
)
INFERENCE_ADDRESS: str = f"tcp://{SERVER_HOST}:{INFERENCE_PORT}"

print(f"trainer:   {TRAINER_ADDRESS}")
print(f"inference: {INFERENCE_ADDRESS}")
print(f"model:     {MODEL_ID}")

## Connect

This verifies that the training server is reachable, no other online-learning
session is running, and the warm-start model is available. If a check fails,
ask the backend operator to correct it before continuing.

In [ ]:
robot: sdk_client.Robot = sdk_client.Robot(
    server_address=f"tcp://{ROBOT_HOST}:{rpc_api.DEFAULT_PORT}",
    query_server_address=(
        f"tcp://{ROBOT_HOST}:{rpc_api.DEFAULT_QUERY_PORT}"
    ),
    training_server_address=TRAINER_ADDRESS,
)
trainer: sdk_client.TrainerClient = robot.trainer

current_status: rpc_api.TrainingStatusResponse = (
    trainer.get_online_learning_status()
)
if current_status.phase == "training":
  raise RuntimeError(
      "An online-learning session is already running. Monitor or stop that "
      "session before starting another."
  )

available_model_ids: set[str] = {
    model["model_id"] for model in trainer.list_models()
}
if MODEL_ID not in available_model_ids:
  raise RuntimeError(
      f"The warm-start model is unavailable on the training server: {MODEL_ID}"
  )

print("training server and warm-start model are ready")

## Start online learning

The server derives the dataset, checkpoints, exported model, and inference
snapshot from the warm-start model ID. The robot backend is then attached to
forward newly saved episodes to the session, keyed by the model name the
server returns. If attaching fails, the session is cancelled so the training
server is not left running with nothing feeding it.

In [ ]:
config_overrides: dict[str, object] = {}
if DISABLE_NEW_AUGMENTATIONS:
  config_overrides = {
      "data.aug.hue_max_delta": 0.0,
      "data.aug.color_temp_max": 0,
      "data.aug.joint_angle": 0.0,
      "data.aug.random_crop_min_scale": 1.0,
  }

response: rpc_api.StartSkillTrainingResponse = trainer.start_online_learning(
    init_from_model_id=MODEL_ID,
    inference_gpu=INFERENCE_GPU,
    inference_port=INFERENCE_PORT,
    use_joint_torques=USE_JOINT_TORQUES,
    restart_online_learning=RESTART_TRAINING,
    config_overrides=config_overrides,
    timeout=240_000,
)
if response.error is not None:
  raise RuntimeError(response.error)
if response.online_learning_inference_address is None:
  raise RuntimeError(
      "Online learning started, but the inference service was not launched. "
      "Ask the backend operator to inspect the training server."
  )
if response.online_learning_model_name is None:
  raise RuntimeError(
      "The training server did not report the session's model name. "
      "Ask the backend operator to update it."
  )

forwarding: rpc_api.OnlineEpisodeForwardingStateResponse = (
    robot.online_episode_forwarding.start(response.online_learning_model_name)
)
if forwarding.error is not None:
  trainer.cancel_online_learning()
  raise RuntimeError(forwarding.error)

print("online learning started")
print(
    f"episode forwarding attached ({forwarding.generation or 'no generation'})"
)
print(
    f"configure the policy service for online corrections as {INFERENCE_ADDRESS}"
)

## Collect episodes

The robot backend is attached to the active online-learning session.
Configure online corrections with the inference address printed above and
save at least one usable episode through the normal client interface.

## Verify online learning

Run the following cell after saving an episode, and rerun it whenever you want
an updated view. A live session initially reports zero steps while waiting for
data. Online learning is confirmed once `steps_completed` increases between
checks. The exported model ID is stable for the session while its weights are
updated in place.

In [ ]:
status: rpc_api.TrainingStatusResponse = trainer.get_online_learning_status()

if status.error is not None:
  raise RuntimeError(status.error)

print(f"phase:      {status.phase}")
print(f"steps:      {status.steps_completed}/{status.max_steps}")
print(f"loss:       {status.loss:.4f}")
print(f"steps/sec:  {status.fps:.2f}")

warehouse_models: list[dict[str, object]] = trainer.list_models()
exported_model: dict[str, object] | None = next(
    (
        model
        for model in warehouse_models
        if model["model_id"] == status.model_id
    ),
    None,
)

if status.model_id is None:
  print("exported model ID is not available yet")
elif exported_model is None:
  print(f"warehouse export pending: {status.model_id}")
else:
  print(f"latest exported model ID: {status.model_id}")
  print(f"exported at: {exported_model['timestamp']}")

if status.phase != "training":
  raise RuntimeError(f"Online learning is not active: {status.phase}")
if status.steps_completed == 0:
  print("waiting for the first usable saved episode")
else:
  print("online learning has completed gradient updates")

## Stop online learning

The training server continues training until the session is stopped. Stop
episode forwarding first, so no episode is saved under a session that has
ended. The backend refuses to stop forwarding while an episode is being
recorded; wait for it to finish and rerun the cell.

In [ ]:
forwarding_state: rpc_api.OnlineEpisodeForwardingStateResponse = (
    robot.online_episode_forwarding.stop()
)
if forwarding_state.error is not None:
  raise RuntimeError(forwarding_state.error)

cancelled: rpc_api.CancelTrainingResponse = trainer.cancel_online_learning()
if cancelled.error is not None:
  raise RuntimeError(cancelled.error)